This notebook reads the csv produce by Toky and:
- remove ontologies not included in the experiments
- for each remaining ontology, remove question types (i.e. questions) when the number of corresponding questions is less than a mininum number

The results is stored and further processed to sample the k examples used for in-context learning.

In [1]:
import pandas as pd

df = pd.read_csv("../dataset/qgenllm-updated-20260209.csv", sep=";")
print(df.shape)

display(df.head())

print("ONTOLOGIES")
print(df.ontology.value_counts())

(513985, 10)


,axiom_pattern,axiom,template,question,question_type,op_category,ontology,file,question_type_short,op_category_short
0,[X] SubclassOf([prop] some [Y]),CarnivorousPlant SubclassOf(eats some Animal),Does [X] [prop/*OP_VERB-infinitive] some [Y]?,Does a carnivorous plant eat some animal?,op_Yes-No-2-part-1-rel+1-quant-some,OP_VERB,AWO,Yes-No-2-part-1-rel+1-quant-some.csv,yn2p1r_1_qs,verb
1,[X] SubclassOf([prop] some [Y]),Warthog SubclassOf(eats some Grass),Does [X] [prop/*OP_VERB-infinitive] some [Y]?,Does a warthog eat some grass?,op_Yes-No-2-part-1-rel+1-quant-some,OP_VERB,AWO,Yes-No-2-part-1-rel+1-quant-some.csv,yn2p1r_1_qs,verb
2,[X] SubclassOf([prop] some [Y]),Warthog SubclassOf(eats some Gras),Does [X] [prop/*OP_VERB-infinitive] some [Y]?,Does a warthog eat some gras?,op_Yes-No-2-part-1-rel+1-quant-some,OP_VERB,AWO,Yes-No-2-part-1-rel+1-quant-some.csv,yn2p1r_1_qs,verb
3,[X] SubclassOf([prop] some [Y]),Warthog SubclassOf(eats some PlantParts),Does [X] [prop/*OP_VERB-infinitive] some [Y]?,Does a warthog eat some plant parts?,op_Yes-No-2-part-1-rel+1-quant-some,OP_VERB,AWO,Yes-No-2-part-1-rel+1-quant-some.csv,yn2p1r_1_qs,verb
4,[X] SubclassOf([prop] some [Y]),Warthog SubclassOf(eats some Root),Does [X] [prop/*OP_VERB-infinitive] some [Y]?,Does a warthog eat some root?,op_Yes-No-2-part-1-rel+1-quant-some,OP_VERB,AWO,Yes-No-2-part-1-rel+1-quant-some.csv,yn2p1r_1_qs,verb


ONTOLOGIES
ontology
Stuff           340412
Pizza            62701
BioTop           61835
Cultural-On      22668
swo              19257
exmo              4009
MFOEM             1327
AWO               1153
CopyrightAll       302
bcio               249
olia                72
Name: count, dtype: int64


FIX NAME OF THE QUESTION TYPES. REMOVE "_op"


In [2]:
df["question_type"] = df["question_type"].str.replace("op_", "", regex=False)

Remove ontologies with questions not suited for the LLM-based question generation task.

In [3]:
print("UNIQUE ONTOLOGIES")
print(df['ontology'].unique())
n_rows1 = df.shape[0]

remove_ontologies = ["MFOEM", "bcio", "exmo", "swo"]

print("SAMPLE OF THE ONTOLOGIES THAT WILL BE REMOVED")
display( df[df.ontology.isin(remove_ontologies)].groupby("ontology")[["ontology", "question_type", "question"]].sample(2) )
df_filtered = df[~df.ontology.isin(remove_ontologies)]
print(df_filtered.shape)

df_filtered.to_csv("../dataset/qgenllm-updated-2-filtered.csv", sep=";", index=False)
n_rows2 = df_filtered.shape[0]

print(f"Removed {n_rows1 - n_rows2} rows.")

print("UNIQUE ONTOLOGIES AFTER REMOVAL")
print(df_filtered['ontology'].unique())

assert (set(df_filtered['ontology'].unique()) & set(remove_ontologies)) == set(), "Some ontologies that should have been removed are still present in the filtered dataset."

UNIQUE ONTOLOGIES
['AWO' 'bcio' 'BioTop' 'CopyrightAll' 'Cultural-On' 'exmo' 'MFOEM' 'olia'
 'Pizza' 'Stuff' 'swo']
SAMPLE OF THE ONTOLOGIES THAT WILL BE REMOVED


,ontology,question_type,question
90483,MFOEM,What-2-part-1-rel-quant-some,Which mental process has an occurrent part tha...
91306,MFOEM,What-2-part-1-rel,Which entity has an occurrent part that is a n...
1174,bcio,Yes-No-2-part-1-rel,Does a supervision of person source have a par...
1302,bcio,What-2-part-1-rel,Which entity has a participant that is a perso...
89413,exmo,Definition,What is a one dimensional continuant fiat boun...
88870,exmo,Definition,What is a warm up exercise?
507130,swo,What-2-part-1-rel-quant-some,Which entity has a clause that is some time fo...
495678,swo,What-2-part-1-rel-quant-only,Which entity is realized in only a planned pro...


(489143, 10)
Removed 24842 rows.
UNIQUE ONTOLOGIES AFTER REMOVAL
['AWO' 'BioTop' 'CopyrightAll' 'Cultural-On' 'olia' 'Pizza' 'Stuff']


For each ontology, remove the question types associated to less than < N_MIN

In [4]:
N_MIN = 10

df_final = pd.DataFrame()
dfs = []

print(f"Removing question types with < {N_MIN} examples - per ontology")

for ontology, group in df_filtered.groupby("ontology"):
    print()
    print(ontology)
    counts = group['question_type'].value_counts()
    to_keep = counts[counts >= N_MIN].index
    remove_qt = counts[counts < N_MIN].index

    print("KEEPING:", list(to_keep))
    print("REMOVING:", list(remove_qt))
    print(f"Ontology: {ontology}")
    print(f"  Removing question types: {list(remove_qt)}")
    
    df_ontology_filtered = group[group['question_type'].isin(to_keep)]
    dfs.append(df_ontology_filtered)
    print(f"Ontology {ontology}: {group.shape[0]} -> {df_ontology_filtered.shape[0]} rows")
df_final = pd.concat(dfs)

df_final = df_final.sort_values(by=["ontology", "question_type"]).reset_index(drop=True)
df_final["row_id"] = list(range(len(df_final)))


n_rows3 = df_final.shape[0]
print()
print()
print(f"Removed {n_rows2 - n_rows3} rows due to few examples per question type.")

# This is the dataset used for the experiments
filename = "../dataset/qgenllm-updated-2-filtered-min_q.csv"
df_final.to_csv(filename, sep=";", index=False)

print()
print("Saved:", filename)
print("all done - notebook ends here")

Removing question types with < 10 examples - per ontology

AWO
KEEPING: ['Yes-No-2-part-1-rel', 'Yes-No-2-part-1-rel+1-quant-some', 'What-2-part-1-rel', 'What-2-part-1-rel-quant-some', 'What-1-part-1-rel', 'Yes-No-2-part-1-rel+1-quant-only', 'Definition', 'What-2-part-1-rel-quant-only']
REMOVING: []
Ontology: AWO
  Removing question types: []
Ontology AWO: 1153 -> 1153 rows

BioTop
KEEPING: ['Yes-No-2-part-1-rel', 'Yes-No-2-part-1-rel+1-quant-only', 'What-1-part-1-rel', 'What-2-part-1-rel', 'What-2-part-1-rel-quant-only', 'Yes-No-2-part-1-rel+1-quant-some', 'What-2-part-1-rel-quant-some', 'Definition']
REMOVING: []
Ontology: BioTop
  Removing question types: []
Ontology BioTop: 61835 -> 61835 rows

CopyrightAll
KEEPING: ['Yes-No-2-part-1-rel', 'What-2-part-1-rel', 'Yes-No-2-part-1-rel+1-quant-only', 'What-2-part-1-rel-quant-only', 'What-1-part-1-rel', 'What-2-part-1-rel-quant-some']
REMOVING: ['Yes-No-2-part-1-rel+1-quant-some']
Ontology: CopyrightAll
  Removing question types: ['Yes-N